# Fairness Mitigation Techniques Comparison

This notebook compares 8 fairness mitigation techniques using the `fairxai.fairness.mitigation` module.

**Techniques**:
- **Pre-processing**: SMOTE, ROS, RUS, ADASYN, Reweighting
- **In-processing**: Exponentiated Gradient, Grid Search  
- **Post-processing**: Threshold Optimizer

**Goal**: Find techniques that improve fairness while maintaining clinical performance (recall ≥ 70%).

In [1]:
# Standard libraries
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Add project to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

# Import FairXAI modules
from fairxai.data.loaders import CardiacDataLoader
from fairxai.models.baseline import BaselineLogisticRegression
from fairxai.fairness.metrics import FairnessMetrics
from fairxai.fairness.mitigation import (
    PreProcessingMitigation,
    InProcessingMitigation,
    PostProcessingMitigation,
    MitigationEngine
)
from fairxai.visualization.plots import plot_fairness_heatmap, plot_model_performance

print(f"✓ Imports successful")
print(f"Project root: {project_root}")

✓ Imports successful
Project root: /home/miguel/Desktop/Dissertacao/Code/FairXAI


## 1. Load Processed Datasets

Load train/test splits from preprocessed data.

In [2]:
# Load Cleveland dataset
dataset_name = 'cleveland'
data_dir = project_root / 'data/processed/cardiac'

# Check if data exists
train_file = data_dir / f'{dataset_name}_train.csv'
test_file = data_dir / f'{dataset_name}_test.csv'

if not train_file.exists() or not test_file.exists():
    print(f"❌ Processed data not found!")
    print(f"   Expected: {train_file}")
    print(f"   Please run: bash scripts/cardiac_pipeline.sh")
    raise FileNotFoundError("Run pipeline first to generate processed data")

# Load combined train/test files
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

# Encode sex if categorical
if train_df['sex'].dtype == 'object':
    sex_map = {'Male': 1, 'Female': 0, 'M': 1, 'F': 0}
    train_df['sex'] = train_df['sex'].map(sex_map)
    test_df['sex'] = test_df['sex'].map(sex_map)
    print("✓ Encoded categorical 'sex' variable")

# Separate features, target, and sensitive attributes
# Exclude target, sensitive attrs, metadata, and original categorical columns
exclude_cols = [
    'heart_disease', 'age_group', 'sex', 'sex_extended', 'sex_bin',
    'Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope',
    '_dataset_source', '_dataset_file', 'age_raw', 'HeartDisease'
]
exclude_cols = [col for col in exclude_cols if col in train_df.columns]

X_train = train_df.drop(columns=exclude_cols)
y_train = train_df['heart_disease']
sensitive_train = train_df[['age_group', 'sex']]

X_test = test_df.drop(columns=exclude_cols)
y_test = test_df['heart_disease']
sensitive_test = test_df[['age_group', 'sex']]

print(f"✓ Loaded {dataset_name}")
print(f"  Train: {X_train.shape}, Test: {X_test.shape}")
print(f"  Features: {list(X_train.columns)[:5]}... ({X_train.shape[1]} total)")
print(f"  Class balance (train): {y_train.value_counts().to_dict()}")
print(f"  Class balance (test): {y_test.value_counts().to_dict()}")

✓ Encoded categorical 'sex' variable
✓ Loaded cleveland
  Train: (206, 13), Test: (89, 13)
  Features: ['age', 'cp', 'trestbps', 'chol', 'fbs']... (13 total)
  Class balance (train): {0: 112, 1: 94}
  Class balance (test): {0: 48, 1: 41}


## 2. Train Baseline Model

Baseline Logistic Regression without mitigation.

In [4]:
# Train baseline
baseline_model = BaselineLogisticRegression(random_state=42)
#baseline_model.fit(X_train, y_train)
baseline_model.train(X_train, y_train)

# Evaluate
baseline_metrics = baseline_model.evaluate(X_test, y_test)
print("Baseline Performance:")
for metric, value in baseline_metrics.items():
    print(f"  {metric}: {value:.3f}")

# Assess fairness
y_pred_baseline = baseline_model.predict(X_test)
y_proba_baseline = baseline_model.predict_proba(X_test)[:, 1]

predictions_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_pred_baseline,
    'y_proba': y_proba_baseline,
    'age_group': sensitive_test['age_group'].values,
    'sex': sensitive_test['sex'].values
})

fairness_calculator = FairnessMetrics()
baseline_fairness = fairness_calculator.calculate_all_metrics(
    predictions_df,
    sensitive_attributes=['sex', 'age_group'],
    privileged_groups={'sex': 1, 'age_group': '60-69'}
)

print("\nBaseline Fairness:")
print(f"  Demographic Parity (sex): {baseline_fairness['demographic_parity']['sex']['max_difference']:.3f}")
print(f"  Demographic Parity (age): {baseline_fairness['demographic_parity']['age_group']['max_difference']:.3f}")

Baseline Performance:
  accuracy: 1.000
  precision: 1.000
  recall: 1.000
  f1_score: 1.000
  auc_roc: 1.000
  n_samples: 89.000
  n_positive: 41.000
  n_negative: 48.000
  threshold: 0.500


TypeError: unsupported format string passed to dict.__format__

## 3. Test Mitigation Techniques

Apply all 8 techniques using MitigationEngine.

In [ ]:
# Initialize engine
engine = MitigationEngine()

# Define techniques to test
techniques_config = {
    # Pre-processing
    'smote': {'stage': 'pre-processing', 'params': {'k_neighbors': 5}},
    'ros': {'stage': 'pre-processing', 'params': {}},
    'rus': {'stage': 'pre-processing', 'params': {}},
    'adasyn': {'stage': 'pre-processing', 'params': {}},
    'reweighting': {'stage': 'pre-processing', 'params': {}},
    # In-processing
    'exponentiated_gradient': {'stage': 'in-processing', 'params': {'constraint_type': 'equalized_odds'}},
    'grid_search': {'stage': 'in-processing', 'params': {'constraint_type': 'equalized_odds'}},
    # Post-processing
    'threshold_optimizer': {'stage': 'post-processing', 'params': {'constraint_type': 'equalized_odds'}}
}

# Run all techniques
all_results = []

print("Testing mitigation techniques:\n")
for technique_name, config in techniques_config.items():
    print(f"  {technique_name}...", end=' ')
    
    try:
        result = engine.apply_technique(
            technique_name=technique_name,
            stage=config['stage'],
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            sensitive_train=sensitive_train,
            sensitive_test=sensitive_test,
            **config.get('params', {})
        )
        
        # Store result
        all_results.append({
            'technique': technique_name,
            'stage': config['stage'],
            **result['test_metrics'],
            'fairness': result.get('fairness', {})
        })
        
        print(f"✓ (acc: {result['test_metrics']['accuracy']:.3f}, recall: {result['test_metrics']['recall']:.3f})")
        
    except Exception as e:
        print(f"✗ Error: {e}")

print(f"\n✓ Completed {len(all_results)} techniques")

## 4. Create Comparison Table

In [ ]:
# Create DataFrame
comparison_df = pd.DataFrame(all_results)

# Add baseline for comparison
baseline_row = {
    'technique': 'baseline',
    'stage': 'none',
    **baseline_metrics,
    'fairness': baseline_fairness
}
comparison_full = pd.concat([pd.DataFrame([baseline_row]), comparison_df], ignore_index=True)

# Extract fairness metrics
comparison_full['dp_sex'] = comparison_full['fairness'].apply(
    lambda x: x.get('demographic_parity', {}).get('sex', {}).get('max_difference', np.nan)
)
comparison_full['dp_age'] = comparison_full['fairness'].apply(
    lambda x: x.get('demographic_parity', {}).get('age_group', {}).get('max_difference', np.nan)
)

# Select key columns
display_cols = ['technique', 'stage', 'accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'dp_sex', 'dp_age']
comparison_table = comparison_full[display_cols].copy()

print("Mitigation Comparison:")
print("="*120)
print(comparison_table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

## 5. Clinical Constraint Check

Filter techniques meeting recall ≥ 0.70.

In [ ]:
# Apply clinical constraint
RECALL_THRESHOLD = 0.70

meets_clinical = comparison_table[comparison_table['recall'] >= RECALL_THRESHOLD].copy()

print(f"Techniques meeting clinical constraint (recall ≥ {RECALL_THRESHOLD}):")
print("="*120)
print(meets_clinical.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f"\n✓ {len(meets_clinical)} of {len(comparison_table)} techniques meet clinical requirements")

## 6. Visualizations

In [ ]:
# Performance vs Fairness trade-off
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy vs DP (sex)
ax1 = axes[0]
scatter = ax1.scatter(
    comparison_table['dp_sex'], 
    comparison_table['accuracy'],
    c=comparison_table['recall'],
    s=150,
    alpha=0.6,
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=1
)
for idx, row in comparison_table.iterrows():
    ax1.annotate(
        row['technique'][:6],
        (row['dp_sex'], row['accuracy']),
        fontsize=8,
        ha='center'
    )

ax1.set_xlabel('Demographic Parity Difference (Sex)', fontsize=11)
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('Accuracy vs Fairness (Sex)', fontsize=12, fontweight='bold')
ax1.axhline(y=0.70, color='red', linestyle='--', alpha=0.3, label='Min Recall')
ax1.grid(alpha=0.3)
cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('Recall', fontsize=10)

# Recall vs DP (age)
ax2 = axes[1]
scatter2 = ax2.scatter(
    comparison_table['dp_age'],
    comparison_table['recall'],
    c=comparison_table['accuracy'],
    s=150,
    alpha=0.6,
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=1
)
for idx, row in comparison_table.iterrows():
    ax2.annotate(
        row['technique'][:6],
        (row['dp_age'], row['recall']),
        fontsize=8,
        ha='center'
    )

ax2.set_xlabel('Demographic Parity Difference (Age Group)', fontsize=11)
ax2.set_ylabel('Recall', fontsize=11)
ax2.set_title('Recall vs Fairness (Age Group)', fontsize=12, fontweight='bold')
ax2.axhline(y=0.70, color='red', linestyle='--', alpha=0.3, label='Clinical Threshold')
ax2.grid(alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=ax2)
cbar2.set_label('Accuracy', fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Performance vs fairness visualization complete")

## 7. Top Recommendations

Best techniques balancing performance and fairness.

In [ ]:
# Score techniques (higher is better)
# Simple scoring: minimize DP, maximize recall, maintain accuracy
meets_clinical['fairness_score'] = 1 - (meets_clinical['dp_sex'] + meets_clinical['dp_age']) / 2
meets_clinical['performance_score'] = (meets_clinical['recall'] + meets_clinical['accuracy']) / 2
meets_clinical['overall_score'] = (meets_clinical['fairness_score'] + meets_clinical['performance_score']) / 2

# Sort by overall score
top_techniques = meets_clinical.sort_values('overall_score', ascending=False)

print("Top 5 Recommended Techniques:")
print("="*120)
display_cols_top = ['technique', 'stage', 'recall', 'accuracy', 'dp_sex', 'dp_age', 'overall_score']
print(top_techniques[display_cols_top].head().to_string(index=False, float_format=lambda x: f'{x:.3f}'))

## Summary

**Key Findings:**

1. **Clinical Performance**: X techniques meet the recall ≥ 70% threshold
2. **Fairness Impact**: Most effective techniques are in [stage] category
3. **Trade-offs**: Some techniques improve fairness but reduce performance
4. **Recommendation**: [Top technique] balances all constraints

**Next Steps:**
- Apply top technique in production pipeline
- Conduct statistical significance testing
- Evaluate on additional datasets
- Test ensemble approaches